In [47]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"

In [54]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

from src.utils.readers import CsvReader

builder = (
        SparkSession.builder
        .appName("bronze_wt_power")
        .master("local[*]")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
    )

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

RAW_PATH = "/Users/weeliptan/Documents/Repos/ColibriLocal/data/raw/wind_turbines/*"

reader = CsvReader(spark)
df = reader.read(RAW_PATH)

In [69]:
df.show()

+-------------------+----------+----------+--------------+------------+
|          timestamp|turbine_id|wind_speed|wind_direction|power_output|
+-------------------+----------+----------+--------------+------------+
|2022-03-01 00:00:00|        11|       9.1|           269|         2.9|
|2022-03-01 00:00:00|        12|      11.3|           316|         2.5|
|2022-03-01 00:00:00|        13|      11.2|           148|         3.7|
|2022-03-01 00:00:00|        14|      10.7|            97|         1.6|
|2022-03-01 00:00:00|        15|      11.0|            81|         4.4|
|2022-03-01 01:00:00|        11|      12.3|           245|         1.8|
|2022-03-01 01:00:00|        12|      11.0|           293|         2.2|
|2022-03-01 01:00:00|        13|      11.4|           270|         1.9|
|2022-03-01 01:00:00|        14|      10.4|           140|         2.3|
|2022-03-01 01:00:00|        15|      14.6|           283|         4.3|
|2022-03-01 02:00:00|        11|      14.3|           135|      

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql import Window

w = Window.partitionBy("turbine_id").orderBy("timestamp")

df_gaps = (
    df
    .withColumn("prev_timestamp", F.lag("timestamp").over(w))
    .withColumn("diff_hours", 
    (F.unix_timestamp("timestamp") - F.unix_timestamp("prev_timestamp")) / 3600
    )
)

# identify
df_gaps.filter(F.col("diff_hours") != 1).show()

# Determine start and end date
# Assumptions
# 1. wind direction ranges from 0 to 360
df_gaps.groupBy("turbine_id").agg(
    F.min("timestamp").alias("min_timestamp"),
    F.max("timestamp").alias("max_timestamp"),
    F.count("turbine_id").alias("hourly_entries"),
    F.sum((F.col("diff_hours") - 1).cast("int")).alias("missing_entries"),
    F.min("wind_direction").alias("wd_min"),
    F.max("wind_direction").alias("wd_max"),
    F.sum(F.when(F.col("wind_direction").isNull(), 1).otherwise(0)).alias("wd_null"),
    F.min("wind_speed").alias("ws_min"),
    F.max("wind_speed").alias("ws_max"),
    F.sum(F.when(F.col("wind_speed").isNull(), 1).otherwise(0)).alias("ws_null"),
    F.min("power_output").alias("pwr_min"),
    F.max("power_output").alias("pwr_max"),
    F.sum(F.when(F.col("power_output").isNull(), 1).otherwise(0)).alias("pwr_null")
).orderBy("turbine_id").show()

+---------+----------+----------+--------------+------------+--------------+----------+
|timestamp|turbine_id|wind_speed|wind_direction|power_output|prev_timestamp|diff_hours|
+---------+----------+----------+--------------+------------+--------------+----------+
+---------+----------+----------+--------------+------------+--------------+----------+

+----------+-------------------+-------------------+--------------+---------------+------+------+-------+------+------+-------+-------+-------+--------+
|turbine_id|      min_timestamp|      max_timestamp|hourly_entries|missing_entries|wd_min|wd_max|wd_null|ws_min|ws_max|ws_null|pwr_min|pwr_max|pwr_null|
+----------+-------------------+-------------------+--------------+---------------+------+------+-------+------+------+-------+-------+-------+--------+
|         1|2022-03-01 00:00:00|2022-03-31 23:00:00|           744|              0|     0|   359|      0|   9.0|  15.0|      0|    1.5|    4.5|       0|
|         2|2022-03-01 00:00:00|202

26/03/08 21:34:23 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE

In [63]:
df_gaps.filter(F.col("power_output") == 0).show()

+---------+----------+----------+--------------+------------+--------------+----------+
|timestamp|turbine_id|wind_speed|wind_direction|power_output|prev_timestamp|diff_hours|
+---------+----------+----------+--------------+------------+--------------+----------+
+---------+----------+----------+--------------+------------+--------------+----------+



In [49]:
df_gaps.filter(F.col("turbine_id") == 1) \
    .filter(F.col("timestamp").between("2022-03-27 00:00:00", "2022-03-27 23:00:00") ) \
    .show()

+-------------------+----------+----------+--------------+------------+-------------------+----------+
|          timestamp|turbine_id|wind_speed|wind_direction|power_output|     prev_timestamp|diff_hours|
+-------------------+----------+----------+--------------+------------+-------------------+----------+
|2022-03-27 00:00:00|         1|      12.1|            79|         3.4|2022-03-26 23:00:00|       1.0|
|2022-03-27 02:00:00|         1|      11.6|           340|         4.2|2022-03-27 00:00:00|       1.0|
|2022-03-27 02:00:00|         1|      10.9|           175|         1.9|2022-03-27 02:00:00|       0.0|
|2022-03-27 03:00:00|         1|      11.4|           130|         2.1|2022-03-27 02:00:00|       1.0|
|2022-03-27 04:00:00|         1|      13.4|           187|         4.0|2022-03-27 03:00:00|       1.0|
|2022-03-27 05:00:00|         1|      12.7|           334|         4.0|2022-03-27 04:00:00|       1.0|
|2022-03-27 06:00:00|         1|      14.7|           136|         2.3|20

+-------------------+----------+----------+--------------+------------+-------------------+----------+
|          timestamp|turbine_id|wind_speed|wind_direction|power_output|     prev_timestamp|diff_hours|
+-------------------+----------+----------+--------------+------------+-------------------+----------+
|2022-03-27 02:00:00|         1|      10.9|           175|         1.9|2022-03-27 02:00:00|       0.0|
|2022-03-27 02:00:00|         2|       9.5|            94|         4.4|2022-03-27 02:00:00|       0.0|
|2022-03-27 02:00:00|         3|      13.4|           110|         3.0|2022-03-27 02:00:00|       0.0|
|2022-03-27 02:00:00|         4|      14.8|           217|         3.3|2022-03-27 02:00:00|       0.0|
|2022-03-27 02:00:00|         5|      14.4|           156|         4.3|2022-03-27 02:00:00|       0.0|
|2022-03-27 02:00:00|         6|      12.3|           114|         4.4|2022-03-27 02:00:00|       0.0|
|2022-03-27 02:00:00|         7|      10.7|           274|         2.5|20

In [35]:
print(df.count())
print(df.distinct().count())

11160
11160


In [41]:
print(df.select("turbine_id", "timestamp").count())
print(df.select("turbine_id", "timestamp").distinct().count())

# on the 27th of Mar 2022 - 2am entry seems to have duplicate entries 
df.select("turbine_id", "timestamp").groupBy("turbine_id", "timestamp").count().orderBy(F.col("count").desc(), F.col("turbine_id")).show()

11160
11145
+----------+-------------------+-----+
|turbine_id|          timestamp|count|
+----------+-------------------+-----+
|         1|2022-03-27 02:00:00|    2|
|         2|2022-03-27 02:00:00|    2|
|         3|2022-03-27 02:00:00|    2|
|         4|2022-03-27 02:00:00|    2|
|         5|2022-03-27 02:00:00|    2|
|         6|2022-03-27 02:00:00|    2|
|         7|2022-03-27 02:00:00|    2|
|         8|2022-03-27 02:00:00|    2|
|         9|2022-03-27 02:00:00|    2|
|        10|2022-03-27 02:00:00|    2|
|        11|2022-03-27 02:00:00|    2|
|        12|2022-03-27 02:00:00|    2|
|        13|2022-03-27 02:00:00|    2|
|        14|2022-03-27 02:00:00|    2|
|        15|2022-03-27 02:00:00|    2|
|         1|2022-03-01 08:00:00|    1|
|         1|2022-03-31 16:00:00|    1|
|         1|2022-03-31 15:00:00|    1|
|         1|2022-03-12 02:00:00|    1|
|         1|2022-03-10 12:00:00|    1|
+----------+-------------------+-----+
only showing top 20 rows



In [42]:
df \
    .filter(F.col("turbine_id") == 1) \
    .filter(F.col("timestamp") == "2022-03-27 02:00:00") \
    .show()

+-------------------+----------+----------+--------------+------------+
|          timestamp|turbine_id|wind_speed|wind_direction|power_output|
+-------------------+----------+----------+--------------+------------+
|2022-03-27 02:00:00|         1|      11.6|           340|         4.2|
|2022-03-27 02:00:00|         1|      10.9|           175|         1.9|
+-------------------+----------+----------+--------------+------------+



In [46]:
df \
    .filter(F.col("turbine_id") == 2) \
    .filter(F.col("timestamp").between("2022-03-27 00:00:00", "2022-03-27 23:00:00") ) \
    .show()

+-------------------+----------+----------+--------------+------------+
|          timestamp|turbine_id|wind_speed|wind_direction|power_output|
+-------------------+----------+----------+--------------+------------+
|2022-03-27 00:00:00|         2|      10.7|           202|         3.7|
|2022-03-27 02:00:00|         2|      12.3|           233|         4.1|
|2022-03-27 02:00:00|         2|       9.5|            94|         4.4|
|2022-03-27 03:00:00|         2|       9.8|            82|         3.9|
|2022-03-27 04:00:00|         2|      13.2|           263|         1.5|
|2022-03-27 05:00:00|         2|      12.6|           308|         2.0|
|2022-03-27 06:00:00|         2|      10.6|            96|         4.1|
|2022-03-27 07:00:00|         2|       9.8|             2|         4.1|
|2022-03-27 08:00:00|         2|      11.4|            30|         1.6|
|2022-03-27 09:00:00|         2|      11.2|            20|         4.3|
|2022-03-27 10:00:00|         2|      13.8|           279|      